In [2]:

import matplotlib.pyplot as plt
import pandas as pd
from prettytable import PrettyTable

In [3]:
datos = pd.read_csv('/content/Stress Indicators Dataset for Mental Health Classification.csv')

In [12]:
from prettytable import PrettyTable

def riesgo_academico_por_tipo_estres(df, umbral=4):
    variables = [
        'academic_overload',
        'peer_competition',
        'low_academic_confidence',
        'academic_conflicts',
        'professor_difficulties'
    ]

    nombres_estres = {
        0: 'Distress',
        1: 'Eustress',
        2: 'Mixto/Otro'
    }

    tabla = PrettyTable()
    tabla.field_names = [
        "Tipo de estrés",
        "Total estudiantes",
        "Promedio riesgo académico",
        "Estudiantes en riesgo",
        "% en riesgo"
    ]

    df = df.copy()
    df['riesgo_academico'] = df[variables].mean(axis=1)
    df['en_riesgo'] = df['riesgo_academico'] >= umbral

    resumen = df.groupby('stress_type').agg(
        total=('stress_type', 'count'),
        promedio_riesgo=('riesgo_academico', 'mean'),
        estudiantes_riesgo=('en_riesgo', 'sum')
    )

    for tipo, fila in resumen.iterrows():
        porcentaje = (fila['estudiantes_riesgo'] / fila['total']) * 100

        tabla.add_row([
            nombres_estres.get(tipo, tipo),
            fila['total'],
            f"{fila['promedio_riesgo']:.2f}",
            fila['estudiantes_riesgo'],
            f"{porcentaje:.2f}%"
        ])

    print(tabla)

riesgo_academico_por_tipo_estres(datos)

+----------------+-------------------+---------------------------+-----------------------+-------------+
| Tipo de estrés | Total estudiantes | Promedio riesgo académico | Estudiantes en riesgo | % en riesgo |
+----------------+-------------------+---------------------------+-----------------------+-------------+
|    Distress    |        70.0       |            3.65           |          25.0         |    35.71%   |
|    Eustress    |       1828.0      |            2.55           |          42.0         |    2.30%    |
|   Mixto/Otro   |       102.0       |            1.78           |          0.0          |    0.00%    |
+----------------+-------------------+---------------------------+-----------------------+-------------+


In [14]:
from prettytable import PrettyTable

def perfil_emocional_fisico_por_tipo_estres(df):
    emocionales = [
        'anxiety_tension',
        'sadness_low_mood',
        'loneliness_isolation',
        'irritability',
        'restlessness'
    ]

    fisicos = [
        'headaches',
        'health_issues',
        'weight_changes',
        'heartbeat_palpitations'
    ]

    nombres_estres = {
        0: 'Distress',
        1: 'Eustress',
        2: 'Mixto/Otro'
    }

    tabla = PrettyTable()
    tabla.field_names = [
        "Tipo de estrés",
        "Promedio emocional",
        "Promedio físico",
        "Dimensión dominante"
    ]

    df = df.copy()
    df['puntaje_emocional'] = df[emocionales].mean(axis=1)
    df['puntaje_fisico'] = df[fisicos].mean(axis=1)

    resumen = df.groupby('stress_type')[[
        'puntaje_emocional',
        'puntaje_fisico'
    ]].mean()

    for tipo, fila in resumen.iterrows():
        if fila['puntaje_emocional'] > fila['puntaje_fisico']:
            dominante = "Emocional"
        elif fila['puntaje_fisico'] > fila['puntaje_emocional']:
            dominante = "Físico"
        else:
            dominante = "Equilibrado"

        tabla.add_row([
            nombres_estres.get(tipo, tipo),
            f"{fila['puntaje_emocional']:.2f}",
            f"{fila['puntaje_fisico']:.2f}",
            dominante
        ])

    print(tabla)


perfil_emocional_fisico_por_tipo_estres(datos)

+----------------+--------------------+-----------------+---------------------+
| Tipo de estrés | Promedio emocional | Promedio físico | Dimensión dominante |
+----------------+--------------------+-----------------+---------------------+
|    Distress    |        4.01        |       3.89      |      Emocional      |
|    Eustress    |        2.59        |       2.57      |      Emocional      |
|   Mixto/Otro   |        1.72        |       1.76      |        Físico       |
+----------------+--------------------+-----------------+---------------------+


In [16]:
from prettytable import PrettyTable

def ranking_indicadores_por_tipo_estres(df, top_n=5):
    indicadores = [
        'stress_experience',
        'heartbeat_palpitations',
        'anxiety_tension',
        'sleep_problems',
        'restlessness',
        'irritability',
        'sadness_low_mood',
        'loneliness_isolation',
        'concentration_problems',
        'headaches',
        'health_issues',
        'weight_changes',
        'academic_overload',
        'peer_competition',
        'low_academic_confidence',
        'subject_confidence',
        'academic_conflicts',
        'class_attendance',
        'professor_difficulties',
        'work_environment',
        'home_environment',
        'relationship_stress',
        'lack_relaxation_time'
    ]

    nombres_estres = {
        0: 'Distress',
        1: 'Eustress',
        2: 'Mixto/Otro'
    }

    tabla = PrettyTable()
    tabla.field_names = [
        "Tipo de estrés",
        "Posición",
        "Indicador",
        "Promedio"
    ]

    resumen = df.groupby('stress_type')[indicadores].mean()

    for tipo, fila in resumen.iterrows():
        ranking = fila.sort_values(ascending=False).head(top_n)

        for posicion, (indicador, promedio) in enumerate(ranking.items(), start=1):
            tabla.add_row([
                nombres_estres.get(tipo, tipo),
                posicion,
                indicador,
                f"{promedio:.2f}"
            ])

    print(tabla)


ranking_indicadores_por_tipo_estres(datos, top_n=5)

+----------------+----------+------------------------+----------+
| Tipo de estrés | Posición |       Indicador        | Promedio |
+----------------+----------+------------------------+----------+
|    Distress    |    1     |      restlessness      |   4.44   |
|    Distress    |    2     |      irritability      |   4.40   |
|    Distress    |    3     |       headaches        |   4.37   |
|    Distress    |    4     | concentration_problems |   4.13   |
|    Distress    |    5     | heartbeat_palpitations |   4.10   |
|    Eustress    |    1     |    class_attendance    |   3.28   |
|    Eustress    |    2     |   stress_experience    |   2.98   |
|    Eustress    |    3     |     sleep_problems     |   2.78   |
|    Eustress    |    4     |   academic_conflicts   |   2.75   |
|    Eustress    |    5     | heartbeat_palpitations |   2.74   |
|   Mixto/Otro   |    1     |    class_attendance    |   2.48   |
|   Mixto/Otro   |    2     |   stress_experience    |   2.21   |
|   Mixto/